In [1]:
!pip uninstall -y unsloth peft

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [2]:
!pip install unsloth trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 129.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 136.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# 2. Load the BASE model (Notice we do NOT load your adapters)
model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
max_seq_length = 2048

In [3]:
print("Loading raw, un-fine-tuned base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)


Loading raw, un-fine-tuned base model...
==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

In [4]:
# 3. Enable fast inference mode
FastLanguageModel.for_inference(model)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((3072,), eps=1e-05)
        (post_attention_laye

In [10]:
# 4. A much harder, messy, and unstructured test resume
test_resume = """
== CHAD MICHAEL GRIFFIN ==
P: (555) 867-5309 | chaddy.g@email.com
Willing to relocate: Yes | Visa Status: Citizen
[ ABOUT ME ]
Data guru and code wrangler. 10+ years hacking together solutions.

--> STUFF I KNOW:
Py, Pandas, some R, SQL (Postgres mostly), Tableau. AWS (S3, EC2).
Not great at Java but can read it.

--> WHERE I'VE WORKED:
* Data Lead @ FinCorp Analytics
Jan '20 to Present
Did a bunch of ETL stuff. Managed 3 juniors.
* Jr. Analyst - The Startup LLC (2018 - 2020)
Cleaned messy data.

--> SCHOOLING
University of Michigan - class of 17.
Bachelors in Math.
"""

In [14]:
messages = [
    {"role": "system", "content": "You are an expert HR parsing AI. Extract candidate information from the raw resume text and output it as a structured JSON object. Do not include any conversational filler."},
    {"role": "user", "content": f"Extract the profile from this resume text:\n\n{test_resume}"}
]


In [15]:
# Apply standard Phi-3 chat template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

In [16]:

print("\nGenerating response from base model...")

# 5. Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.1,
)


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generating response from base model...


In [17]:

# 6. Decode ONLY the newly generated text (ignoring the prompt)
new_tokens = outputs[0][inputs.shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("\n" + "="*40)
print("--- RAW BASE MODEL OUTPUT ---")
print("="*40)
print(response)
print("="*40)


--- RAW BASE MODEL OUTPUT ---
```json

{

  "name": "Chad Michael Griffin",

  "phone": "(555) 867-5309",

  "email": "chaddy.g@email.com",

  "relocate": "Yes",

  "visa_status": "Citizen",

  "skills": [

    "Python",

    "Pandas",

    "R",

    "SQL",

    "Postgres",

    "Tableau",

    "AWS",

    "S3",

    "EC2"

  ],

  "weaknesses": ["Java"],

  "experience": [

    {

      "title": "Data Lead",

      "company": "FinCorp Analytics",

      "start_date": "Jan '20",

      "end_date": "Present",

      "description": "Managed juniors and did ETL work."

    },

    {

      "title": "Junior Analyst",

      "company": "The Startup LLC",

      "start_date": "2018",

      "end_date": "2020",

      "description": "Cleaned messy data."

    }

  ],

  "education": {

    "institution": "University of Michigan",

    "degree": "Bachelors in Math",

    "graduation_year": "2017"

  }

}

```


In [18]:
# 1. Install dependencies if you restarted a brand new notebook
# !pip install -q unsloth

from unsloth import FastLanguageModel
import torch
import json
import re
# 5. The "Golden JSON" (Ground Truth Answer Key)
ground_truth = {
  "name": "Chad Michael Griffin",
  "phone": "(555) 867-5309",
  "email": "chaddy.g@email.com",
  "visa_status": "Citizen",
  "summary": "Data guru and code wrangler. 10+ years hacking together solutions.",
  "skills": ["Python", "Pandas", "R", "SQL", "PostgreSQL", "Tableau", "AWS", "S3", "EC2", "Java"],
  "experience": [
    {"company": "FinCorp Analytics", "position": "Data Lead", "start_date": "2020-01", "end_date": "Present"},
    {"company": "The Startup LLC", "position": "Jr. Analyst", "start_date": "2018", "end_date": "2020"}
  ],
  "education": [
    {"degree": "Bachelors in Math", "institution": "University of Michigan", "graduation_year": "2017"}
  ]
}

messages = [
    {"role": "system", "content": "You are an expert HR parsing AI. Extract candidate information from the raw resume text and output it as a structured JSON object. Do not include any conversational filler."},
    {"role": "user", "content": f"Extract the profile from this resume text:\n\n{test_resume}"}
]

# Apply standard Phi-3 chat template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

print("\nGenerating response...")

# 6. Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.1,
)

# Decode ONLY the newly generated text
new_tokens = outputs[0][inputs.shape[1]:]
raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("\n--- RAW MODEL OUTPUT ---")
print(raw_response)
print("------------------------\n")


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generating response...

--- RAW MODEL OUTPUT ---
```json

{

  "name": "Chad Michael Griffin",

  "phone": "(555) 867-5309",

  "email": "chaddy.g@email.com",

  "relocate": "Yes",

  "visa_status": "Citizen",

  "skills": [

    "Python",

    "Pandas",

    "R",

    "SQL",

    "Postgres",

    "Tableau",

    "AWS",

    "S3",

    "EC2"

  ],

  "weaknesses": ["Java"],

  "experience": [

    {

      "title": "Data Lead",

      "company": "FinCorp Analytics",

      "start_date": "Jan '20",

      "end_date": "Present",

      "description": "Managed juniors and did ETL work."

    },

    {

      "title": "Junior Analyst",

      "company": "The Startup LLC",

      "start_date": "2018",

      "end_date": "2020",

      "description": "Cleaned messy data."

    }

  ],

  "education": {

    "institution": "University of Michigan",

    "degree": "Bachelors in Math",

    "graduation_year": "2017"

  }

}

```
------------------------



In [23]:
import json
import re

def evaluate_output(raw_text, truth):
    print("\n📊 EVALUATION RESULTS 📊")

    # Extract JSON using regex (split to prevent markdown UI crashing)
    pattern = r'``' + r'`(?:json)?\s*(.*?)\s*``' + r'`'
    json_blocks = re.findall(pattern, raw_text, re.DOTALL)

    if json_blocks:
        clean_json = json_blocks[-1]
    else:
        # Fallback: if the model forgot markdown, try to grab everything from { to }
        match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        clean_json = match.group(0) if match else raw_text

    try:
        parsed_data = json.loads(clean_json)
        print("✅ JSON Validity: Passed (It output valid JSON!)")

        # Calculate Key Extraction Score
        correct_keys = 0
        total_keys = len(truth.keys())

        for key in truth.keys():
            if key in parsed_data:
                correct_keys += 1
            else:
                print(f"  ❌ Missing Schema Key: '{key}'")

        score = (correct_keys / total_keys) * 100
        print(f"🎯 Schema Adherence Score: {score:.1f}% ({correct_keys}/{total_keys} keys found)")

        # Add basic hallucination check
        extra_keys = set(parsed_data.keys()) - set(truth.keys())
        if extra_keys:
            print(f"⚠️ Hallucination Warning: Model invented extra keys: {extra_keys}")

    except json.JSONDecodeError:
        print("❌ JSON Validity: Failed (The model output conversational text or broken formatting)")
        print("🎯 Schema Adherence Score: 0.0% (Could not parse)")

In [24]:
# 3. EXECUTE THE FUNCTION (This is what actually prints the results!)
print("Running evaluation...")
evaluate_output(raw_response, ground_truth)

Running evaluation...

📊 EVALUATION RESULTS 📊
✅ JSON Validity: Passed (It output valid JSON!)
  ❌ Missing Schema Key: 'summary'
🎯 Schema Adherence Score: 87.5% (7/8 keys found)
⚠️ Hallucination Warning: Model invented extra keys: {'weaknesses', 'relocate'}
